In [ ]:
import os
import time
import json
import boto3
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from supabase import create_client

try:
    from dotenv import load_dotenv
except ImportError as exc:
    raise ImportError(
        "python-dotenv is not installed. Run `pip install python-dotenv` and re-run this cell."
    ) from exc

# Load .env.local or .env from repo root (walk up from current dir)
root_env_path = None
cwd = Path.cwd().resolve()
for parent in [cwd, *cwd.parents]:
    for filename in (".env"):
        candidate = parent / filename
        if candidate.exists():
            root_env_path = candidate
            break
    if root_env_path is not None:
        break

if root_env_path is None:
    raise FileNotFoundError(".env.local or .env not found in current or parent directories.")

load_dotenv(root_env_path)

print("Loaded env file:", root_env_path)


Loaded env file: /Users/garrettashcroft/Downloads/travel-app/.env.local


# API

In [3]:
API_KEY = os.environ.get("TA_API_KEY")
print("Loaded TA_API_KEY?", bool(API_KEY))

if not API_KEY:
    raise RuntimeError("Set TA_API_KEY in your environment (keeps key out of the notebook).")

BASE = "https://api.content.tripadvisor.com/api/v1"
HEADERS = {"accept": "application/json"}

CACHE_DIR = Path("ta_cache")
CACHE_DIR.mkdir(exist_ok=True)

def ta_get(path, params=None, sleep_s=0.25, retries=3):
    """GET wrapper with light retry/backoff + polite pacing."""
    params = dict(params or {})
    params["key"] = API_KEY

    url = f"{BASE}{path}"
    last_err = None

    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * (2 ** i))
                continue
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
            time.sleep(sleep_s)
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (2 ** i))

    raise last_err


Loaded TA_API_KEY? True


In [ ]:
API_KEY = os.environ.get("TA_API_KEY")  # recommended: set env var outside notebook
if not API_KEY:
    raise RuntimeError("Set TA_API_KEY in your environment (keeps key out of the notebook).")

BASE = "https://api.content.tripadvisor.com/api/v1"
HEADERS = {"accept": "application/json"}

CACHE_DIR = Path("ta_cache")
CACHE_DIR.mkdir(exist_ok=True)

def ta_get(path, params=None, sleep_s=0.25, retries=3):
    params = dict(params or {})
    params["key"] = API_KEY
    url = f"{BASE}{path}"

    last_err = None
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * (2 ** i))
                continue
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:400]}")
            time.sleep(sleep_s)
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * (2 ** i))
    raise last_err


In [23]:
# Input configuration
USE_BOUNDING_BOXES = True

# Bounding boxes use lat/lon rectangles; step_deg controls grid density (smaller = more calls)
BOUNDING_BOXES = [
    {
        "name": "us_northeast",
        "min_lat": 38.0,
        "max_lat": 43.5,
        "min_lon": -78.0,
        "max_lon": -71.0,
        "step_deg": 1.0,
    },
]

# Optional seed places (used only if USE_BOUNDING_BOXES = False)
SEED_PLACES = []

len(SEED_PLACES), SEED_PLACES[:5]


(79,
 ['Manhattan, New York, USA',
  'Dubai, UAE',
  'Tokyo, Japan',
  'Hong Kong',
  'Singapore'])

In [ ]:
def location_search(search_query, category=None, language="en"):
    params = {"searchQuery": search_query, "language": language}
    if category:
        params["category"] = category
    j = ta_get("/location/search", params=params)
    return j.get("data", [])

def pick_best_geo_result(query, candidates):
    """
    Heuristic: prefer exact-ish name matches and non-zero/valid IDs.
    You can refine this later.
    """
    q = query.lower()
    scored = []
    for c in candidates:
        name = (c.get("name") or "").lower()
        loc_id = c.get("location_id")
        score = 0
        if loc_id: score += 5
        if name and (name in q or q in name): score += 10
        # slight preference if address info exists
        addr = c.get("address_obj") or {}
        if addr.get("country"): score += 1
        if addr.get("city"): score += 1
        scored.append((score, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None

# Build dict of resolved places:
# top_places[location_id] = {"seed": "...", "name": "...", "address_string": "..."}
top_places = {}
unresolved = []

for place in seed_places:
    try:
        candidates = location_search(place, category="geos")
        best = pick_best_geo_result(place, candidates)
        if not best or not best.get("location_id"):
            unresolved.append({"seed": place, "reason": "no_best_match"})
            continue

        loc_id = best["location_id"]
        addr = best.get("address_obj") or {}
        top_places[loc_id] = {
            "seed": place,
            "name": best.get("name"),
            "address_string": addr.get("address_string"),
        }
    except Exception as e:
        unresolved.append({"seed": place, "reason": str(e)})

print("Resolved:", len(top_places))
print("Unresolved:", len(unresolved))
list(top_places.items())[:5]


Resolved: 78
Unresolved: 0


[('28953',
  {'seed': 'Manhattan, New York, USA',
   'name': 'New York',
   'address_string': 'NY'}),
 ('295424',
  {'seed': 'Dubai, UAE',
   'name': 'Dubai',
   'address_string': 'Dubai United Arab Emirates'}),
 ('298184',
  {'seed': 'Tokyo, Japan',
   'name': 'Tokyo',
   'address_string': 'Tokyo Prefecture'}),
 ('7917565',
  {'seed': 'Hong Kong',
   'name': 'Hong Kong Intl Airport',
   'address_string': '1 Sky Plaza Road, Hong Kong China'}),
 ('2146391',
  {'seed': 'Singapore',
   'name': 'Singapore River',
   'address_string': 'Singapore Singapore'})]

In [25]:
with open("top_places_resolved_ids.json", "w", encoding="utf-8") as f:
    json.dump(top_places, f, ensure_ascii=False, indent=2)

"top_places_resolved_ids.json"


'top_places_resolved_ids.json'

In [ ]:
def location_details(location_id, language="en", currency="USD"):
    cache_path = CACHE_DIR / f"details_{location_id}.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text("utf-8"))
    j = ta_get(f"/location/{location_id}/details", params={"language": language, "currency": currency})
    cache_path.write_text(json.dumps(j, ensure_ascii=False, indent=2), encoding="utf-8")
    return j

def nearby_search(lat, lon, category="attractions", radius=10, radius_unit="mi", language="en"):
    j = ta_get("/location/nearby_search", params={
        "latLong": f"{lat},{lon}",
        "category": category,
        "radius": radius,
        "radiusUnit": radius_unit,
        "language": language
    })
    return j.get("data", [])

RADIUS_MILES = 10  # change as needed

attractions_map = {}   # {attraction_id: {"name":..., "seed_place_id":..., "seed_place_name":...}}
seed_errors = []

for seed_geo_id, meta in top_places.items():
    try:
        geo = location_details(seed_geo_id)
        lat = geo.get("latitude")
        lon = geo.get("longitude")
        if lat is None or lon is None:
            seed_errors.append({"seed_geo_id": seed_geo_id, "seed": meta["seed"], "error": "missing lat/lon"})
            continue

        nearby = nearby_search(lat, lon, category="attractions", radius=RADIUS_MILES, radius_unit="mi")
        for item in nearby:
            aid = item.get("location_id")
            if not aid:
                continue
            if aid not in attractions_map:
                attractions_map[aid] = {
                    "name": item.get("name"),
                    "seed_geo_id": seed_geo_id,
                    "seed_place": meta["seed"],
                }
    except Exception as e:
        seed_errors.append({"seed_geo_id": seed_geo_id, "seed": meta["seed"], "error": str(e)})

print("Unique attractions found:", len(attractions_map))
print("Seed geo errors:", len(seed_errors))
list(attractions_map.items())[:5]


Unique attractions found: 742
Seed geo errors: 0


[('33344134',
  {'name': 'Leatherstocking Timber Products',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('9681910',
  {'name': 'C&C Taxi and Airport Transportation',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('19229164',
  {'name': 'Fortin Park',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('27041184',
  {'name': 'Caribbean Day Spa',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'}),
 ('2163160',
  {'name': 'Hanford Mills Museum',
   'seed_geo_id': '28953',
   'seed_place': 'Manhattan, New York, USA'})]

In [ ]:
with open("attractions_ids.json", "w", encoding="utf-8") as f:
    json.dump(attractions_map, f, ensure_ascii=False, indent=2)

"attractions_ids.json"


'attractions_ids.json'

In [ ]:
def flatten_details(d):
    addr = d.get("address_obj") or {}
    return {
        "location_id": d.get("location_id"),
        "name": d.get("name"),
        "web_url": d.get("web_url"),
        "rating": d.get("rating"),
        "num_reviews": d.get("num_reviews"),
        "ranking_string": d.get("ranking_string"),
        "latitude": d.get("latitude"),
        "longitude": d.get("longitude"),
        "address": addr.get("address_string"),
        "city": addr.get("city"),
        "state": addr.get("state"),
        "country": addr.get("country"),
        "phone": d.get("phone"),
        "website": d.get("website"),
        # sometimes present for restaurants; usually not for attractions
        "price_level": d.get("price_level"),
    }

rows = []
detail_errors = []

for aid, meta in attractions_map.items():
    try:
        d = location_details(aid, language="en", currency="USD")
        row = flatten_details(d)
        row["seed_place"] = meta.get("seed_place")
        row["seed_geo_id"] = meta.get("seed_geo_id")
        rows.append(row)
    except Exception as e:
        detail_errors.append({"location_id": aid, "name": meta.get("name"), "error": str(e)})

df = pd.DataFrame(rows)

df.to_csv("attractions.csv", index=False)
df.to_json("attractions.json", orient="records", indent=2, force_ascii=False)

if detail_errors:
    pd.DataFrame(detail_errors).to_csv("attractions_errors.csv", index=False)

print("Saved attractions.csv and attractions.json")
print("Rows:", len(df), "Errors:", len(detail_errors))
df.head()


Saved attractions.csv and attractions.json
Rows: 742 Errors: 0


,location_id,name,web_url,rating,num_reviews,ranking_string,latitude,longitude,address,city,state,country,phone,website,price_level,seed_place,seed_geo_id
0,33344134,Leatherstocking Timber Products,https://www.tripadvisor.com/Attraction_Review-...,5.0,3,None,42.445614,-74.97493,"359 Delaware County Highway 11, West Oneonta, ...",West Oneonta,New York,United States,+1 607-436-9082,https://leatherstockinghandsplits.com/,None,"Manhattan, New York, USA",28953
1,9681910,C&C Taxi and Airport Transportation,https://www.tripadvisor.com/Attraction_Review-...,5.0,1,None,42.44465,-75.03002,"Oneonta, NY",Oneonta,New York,United States,+1 607-434-6531,https://www.facebook.com/candctransport13820/,None,"Manhattan, New York, USA",28953
2,19229164,Fortin Park,https://www.tripadvisor.com/Attraction_Review-...,5.0,2,None,42.453854,-75.013954,"101 Youngs Road, Oneonta, NY 13820",Oneonta,New York,United States,+1 607-432-2900,http://townofoneonta.org/town/parks-recreation...,None,"Manhattan, New York, USA",28953
3,27041184,Caribbean Day Spa,https://www.tripadvisor.com/Attraction_Review-...,None,0,None,42.447132,-75.0235,"5252 State Highway 23, West Oneonta, Oneonta, ...",West Oneonta,New York,United States,+1 607-435-7984,http://www.caribbeandayspa.biz,None,"Manhattan, New York, USA",28953
4,2163160,Hanford Mills Museum,https://www.tripadvisor.com/Attraction_Review-...,4.8,41,None,42.422653,-74.8864,"51 County Highway 12, Meredith, NY 13757-9998",Meredith,New York,United States,+1 607-278-5744,http://www.hanfordmills.org,None,"Manhattan, New York, USA",28953


# Pipeline

In [13]:
# Export TripAdvisor attractions to JSONL and upload to S3 for ai_processor.pyfrom datetime import datetime

# Inputs
ATTRACTIONS_PATH = Path("attractions.json")  # in this notebook directory
S3_BUCKET = os.getenv("S3_BUCKET_NAME")
S3_PREFIX = "raw_scrapes/"

if not S3_BUCKET:
    raise RuntimeError("Set S3_BUCKET_NAME in your env.")

# Load attractions
with open(ATTRACTIONS_PATH, "r", encoding="utf-8") as f:
    attractions = json.load(f)

# Group rows by place (city/country) so each file maps to one place_id

def place_key(row):
    city = (row.get("city") or "").strip()
    country = (row.get("country") or "").strip()
    if city and country:
        return f"{city}, {country}"
    if city:
        return city
    seed = (row.get("seed_place") or "").strip()
    return seed or "Unknown"


def slugify(text):
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in text)
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned.strip("_") or "unknown"


grouped = {}
for a in attractions:
    key = place_key(a)
    grouped.setdefault(key, []).append(a)

s3 = boto3.client("s3")
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for key, rows_for_place in grouped.items():
    lines = []
    for a in rows_for_place:
        title = a.get("name") or "Unknown Attraction"
        city = a.get("city") or ""
        country = a.get("country") or ""
        address = a.get("address") or ""
        rating = a.get("rating") or ""
        reviews = a.get("num_reviews") or ""
        price = a.get("price_level") or ""
        website = a.get("website") or ""
        # Use the grouped place key as the seed to avoid cross-city leakage
        seed_place = key

        content_body = (
            f"Attraction: {title}\n"
            f"Location: {city}, {country}\n"
            f"Detected city: {city}, {country}\n"
            f"Address: {address}\n"
            f"Rating: {rating} (reviews: {reviews})\n"
            f"Price level: {price}\n"
            f"Website: {website}\n"
            f"Seed place: {seed_place}\n"
        ).strip()

        lines.append({
            "source": "TripAdvisor",
            "title": title,
            "url": a.get("web_url") or website or "",
            "content_body": content_body,
            "location_id": a.get("location_id"),
            "seed_place": seed_place,
            "seed_geo_id": a.get("seed_geo_id"),
        })

    file_slug = slugify(key)
    jsonl_dir = Path("jsonl")
    jsonl_dir.mkdir(exist_ok=True)
    jsonl_path = jsonl_dir / f"{file_slug}_{stamp}.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")

    s3_key = f"{S3_PREFIX}{file_slug}_{stamp}.jsonl"
    s3.upload_file(str(jsonl_path), S3_BUCKET, s3_key)
    print("Uploaded:", f"s3://{S3_BUCKET}/{s3_key}")




Uploaded: s3://travel-app-raw-data-private/raw_scrapes/West_Oneonta_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Oneonta_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Meredith_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Delhi_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Bloomville_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Harpersfield_United_States_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Dubai_United_Arab_Emirates_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Marunouchi_Japan_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Chiyoda_Japan_20260211_141514.jsonl
Uploaded: s3://travel-app-raw-data-private/raw_scrapes/Shinjuku_Japan_20260211_141514.jsonl
Uploaded: s3:

In [ ]:
# (Upload handled in previous cell)
    s3_key = f"{S3_PREFIX}{file_slug}_{stamp}.jsonl"
    s3.upload_file(str(jsonl_path), S3_BUCKET, s3_key)
    print("Uploaded:", f"s3://{S3_BUCKET}/{s3_key}")

Uploaded: s3://travel-app-raw-data-private/raw_scrapes/tripadvisor_attractions_20260210_090300.jsonl


In [5]:
# One-time bootstrap: mark existing attractions.json as seen (no S3 upload)
STATE_PATH = Path("ta_state.json")
ATTRACTIONS_PATH = Path("attractions.json")

if not ATTRACTIONS_PATH.exists():
    raise FileNotFoundError("attractions.json not found. Run the scrape first.")

with open(ATTRACTIONS_PATH, "r", encoding="utf-8") as f:
    existing = json.load(f)

seen_ids = sorted({str(r.get("location_id")) for r in existing if r.get("location_id")})
state = {
    "last_run": datetime.now(timezone.utc).isoformat(),
    "seen_attraction_ids": seen_ids,
}
with open(STATE_PATH, "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

print("Bootstrapped ta_state.json with", len(seen_ids), "attractions.")


Bootstrapped ta_state.json with 742 attractions.


In [11]:
# DANGER: Delete all TripAdvisor rows by source_name
from supabase import create_client

SUPABASE_URL = os.environ.get("SUPABASE_URL") or os.environ.get("NEXT_PUBLIC_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY") or os.environ.get("SUPABASE_SERVICE_ROLE")

if not SUPABASE_URL or not SUPABASE_SERVICE_ROLE_KEY:
    raise RuntimeError("Set SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY in your env.")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

# 1) Find TripAdvisor source_id(s)
sources = (
    supabase.table("source")
    .select("source_id, source_name")
    .eq("source_name", "TripAdvisor")
    .execute()
)

source_ids = [row["source_id"] for row in (sources.data or [])]
print("TripAdvisor source_ids:", source_ids)

if not source_ids:
    print("No TripAdvisor sources found. Nothing to delete.")
else:
    # 2) Find attraction_ids linked to TripAdvisor sources
    source_rows = (
        supabase.table("attraction_sources")
        .select("attraction_id, source_id")
        .in_("source_id", source_ids)
        .execute()
    )

    aids = sorted({row["attraction_id"] for row in (source_rows.data or [])})
    print("TripAdvisor-linked attractions:", len(aids))

    # 3) Delete source links first
    supabase.table("attraction_sources").delete().in_("source_id", source_ids).execute()

    # 4) Delete category links and attractions (chunked)
    chunk_size = 500
    for i in range(0, len(aids), chunk_size):
        chunk = aids[i : i + chunk_size]
        supabase.table("attraction_categories").delete().in_("attraction_id", chunk).execute()
        supabase.table("attraction").delete().in_("attraction_id", chunk).execute()

    print("TripAdvisor cleanup complete.")


TripAdvisor source_ids: [108]
TripAdvisor-linked attractions: 30
TripAdvisor cleanup complete.
